# 03 — Feature Engineering

Creates business-meaningful features from the cleaned data. None of these
features use the target (`Churn`) in their computation, so there is no
target leakage. Full implementation lives in `src/feature_engineering.py`
— this notebook demonstrates and validates each feature.

In [1]:
import sys
sys.path.insert(0, '../src')
import pandas as pd
from data_preprocessing import load_raw_data, clean_data
from feature_engineering import engineer_features

raw = load_raw_data('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')
clean = clean_data(raw)
df = engineer_features(clean)
print("Columns after feature engineering:", len(df.columns))
print([c for c in df.columns if c not in clean.columns])


Columns after feature engineering: 32
['TenureGroup', 'MonthlyChargeCategory', 'ServiceCount', 'HasTechSupport', 'HasOnlineSecurity', 'ContractRiskScore', 'IsElectronicCheck', 'CustomerValueSegment']


## Feature 1 — `TenureGroup`
**What:** buckets raw tenure (months) into 5 lifecycle stages.
**Why:** tenure's relationship with churn is non-linear; binning helps
both tree and linear models capture stage-specific risk.
**Business meaning:** flags exactly which lifecycle stage a customer is
in — new/onboarding customers need different treatment than veterans.

In [2]:
print(df['TenureGroup'].value_counts().sort_index())
print()
print((df.groupby('TenureGroup', observed=True)['ChurnFlag'].mean() * 100).round(2))


TenureGroup
0-12 (New)             2186
13-24 (Early)          1024
25-48 (Established)    1594
49-60 (Loyal)           832
61-72 (Veteran)        1407
Name: count, dtype: int64

TenureGroup
0-12 (New)             47.44
13-24 (Early)          28.71
25-48 (Established)    20.39
49-60 (Loyal)          14.42
61-72 (Veteran)         6.61
Name: ChurnFlag, dtype: float64


## Feature 2 — `MonthlyChargeCategory`
**What:** Low/Medium/High tier based on the dataset's own tercile cut
points (33rd/66th percentile), not arbitrary thresholds.
**Why:** easier for stakeholders to reason about than raw dollars; also
gives linear models a simple non-linear signal.
**Business meaning:** flags customers on premium pricing tiers who may be
more price-sensitive.

In [3]:
print(df['MonthlyChargeCategory'].value_counts())
print()
print((df.groupby('MonthlyChargeCategory')['ChurnFlag'].mean() * 100).round(2))


MonthlyChargeCategory
Low       2351
High      2347
Medium    2345
Name: count, dtype: int64

MonthlyChargeCategory
High      34.09
Low       15.87
Medium    29.68
Name: ChurnFlag, dtype: float64


## Feature 3 — `ServiceCount`
**What:** count (0-6) of add-on services subscribed to (security, backup,
device protection, tech support, streaming TV, streaming movies).
**Why:** captures overall "stickiness"/engagement in a single number.
**Business meaning:** low `ServiceCount` = shallow relationship = easier
to leave.

In [4]:
print(df['ServiceCount'].value_counts().sort_index())
print()
print((df.groupby('ServiceCount')['ChurnFlag'].mean() * 100).round(2))


ServiceCount
0    2219
1     966
2    1033
3    1118
4     852
5     571
6     284
Name: count, dtype: int64

ServiceCount
0    21.41
1    45.76
2    35.82
3    27.37
4    22.30
5    12.43
6     5.28
Name: ChurnFlag, dtype: float64


## Features 4 & 5 — `HasTechSupport`, `HasOnlineSecurity`
**What:** explicit binary flags for the two services with the strongest
raw association with retention in EDA.
**Why:** isolates their individual effect clearly, on top of the
aggregate `ServiceCount`.
**Business meaning:** customers without support/security hit friction
alone and are more likely to leave when something goes wrong.

In [5]:
print((df.groupby('HasTechSupport')['ChurnFlag'].mean() * 100).round(2))
print()
print((df.groupby('HasOnlineSecurity')['ChurnFlag'].mean() * 100).round(2))


HasTechSupport
0    31.19
1    15.17
Name: ChurnFlag, dtype: float64

HasOnlineSecurity
0    31.33
1    14.61
Name: ChurnFlag, dtype: float64


## Feature 6 — `ContractRiskScore`
**What:** ordinal risk score (Month-to-month=3 > One year=2 > Two year=1).
**Why:** encodes the natural ordering of commitment level that plain
one-hot encoding would discard.
**Business meaning:** shorter commitment = lower switching cost = higher
churn risk.

In [6]:
print(df[['Contract','ContractRiskScore']].drop_duplicates().sort_values('ContractRiskScore'))


          Contract  ContractRiskScore
11        Two year                  1
1         One year                  2
0   Month-to-month                  3


## Feature 7 — `IsElectronicCheck`
**What:** binary flag for the "Electronic check" payment method
specifically.
**Why:** electronic check is a manual, non-automatic payment method known
to co-occur heavily with churn in this dataset.
**Business meaning:** customers not on autopay have to actively re-decide
to keep paying every cycle, raising reconsideration risk.

In [7]:
print((df.groupby('IsElectronicCheck')['ChurnFlag'].mean() * 100).round(2))


IsElectronicCheck
0    17.06
1    45.29
Name: ChurnFlag, dtype: float64


## Feature 8 — `CustomerValueSegment`
**What:** Low/Medium/High based on tercile cut points of `TotalCharges`
(cumulative revenue to date).
**Why:** distinguishes small/new/cheap accounts from long-standing,
high-value ones — losing the latter is more costly.
**Business meaning:** powers "prioritize retention effort by value" logic
in the dashboard (Revenue at Risk).

In [8]:
print(df['CustomerValueSegment'].value_counts())
print()
print((df.groupby('CustomerValueSegment')['ChurnFlag'].mean() * 100).round(2))


CustomerValueSegment
Low       2348
High      2348
Medium    2347
Name: count, dtype: int64

CustomerValueSegment
High      16.52
Low       39.35
Medium    23.73
Name: ChurnFlag, dtype: float64


## Leakage check

None of the engineered features reference `Churn`/`ChurnFlag` in their
computation — confirmed by inspecting `src/feature_engineering.py`, where
every function only reads from raw customer attributes (`tenure`,
`MonthlyCharges`, `TotalCharges`, service columns, `Contract`,
`PaymentMethod`). `customerID` is excluded from the model's feature list
in `train.py` (`NUMERIC_FEATURES` / `CATEGORICAL_FEATURES` never include
it) since it has no predictive value and is unique per row.

Next: `04_model_training.ipynb`.